# Phase 10 — Final Cross-Project Summary

Pulls together **every trained model evaluated so far across Phases 1-9** into one place:
overall rankings, accuracy-vs-size Pareto fronts, quantization robustness, detection/segmentation
transfer, and the Phase 6 hardware-profiling headline finding.

This is a **synthesis notebook only** — no training, no new `ml/` code. Every number here is
loaded from artifacts each phase's own notebook already produced (`*_summary.json` files and
`*_comparison.csv` tables). See each phase's own notebook (`notebooks/phase_{1..9}_*/`) for the
detailed per-phase analysis and hypothesis testing this summarizes.

**Scope note:** classification (Phases 1-4, 8, 9) shares one top-1/top-5 accuracy metric on Tiny
ImageNet-200 and is merged into a single ranking. Detection (mAP) and segmentation (mIoU) use
different metrics on a different dataset (PASCAL VOC) and are kept as separate sections — they
are not comparable to the classification top-1 numbers or to each other.

## Configuration

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the repository root is found."""
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "models").exists() and (candidate / "results").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))  # so `import ml` resolves regardless of kernel cwd
print(f"Project root: {PROJECT_ROOT}")

RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_DIR = RESULTS_ROOT / "phase_10_final_summary"
FIGURES_DIR = RESULTS_ROOT / "figures_generated" / "phase_10_final_summary"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from ml.plotting import apply_report_style, label_bars
apply_report_style()

# One consistent color per phase, in first-seen order, reused across every chart below.
PHASE_CMAP = plt.get_cmap("tab10")
def phase_colors_for(phases: pd.Series, order: list[str]) -> list:
    return [PHASE_CMAP(order.index(p) % 10) if p in order else "#999999" for p in phases]


## Load Classification Results (Phases 1-4, 8, 9)

Phases 1-4's models are read straight from the per-model `*_summary.json` files each training
notebook already writes (the `make_run_summary` schema) — `experiment_summary.json` (phase-level,
not per-model) and `*_compression_summary.json` (Phase 4's compression-ablation reruns of an
existing architecture, not a new one) are excluded so the ranking doesn't show duplicate/confusing
rows. Phase 9's bypass-ablation model lives under `outputs/pcad/` instead (git-ignored PCAD
output), so it's globbed separately. Phase 8 already has one aggregated CSV with the same column
family.

In [ ]:
def load_summary_jsons(root: Path) -> pd.DataFrame:
    """Every *_summary.json under root, excluding phase-level / compression-ablation ones."""
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for path in sorted(root.rglob("*_summary.json")):
        if path.name == "experiment_summary.json" or path.name.endswith("_compression_summary.json"):
            continue
        try:
            rows.append(json.loads(path.read_text()))
        except Exception as e:
            print(f"[skip] {path}: {e}")
    df = pd.DataFrame(rows)
    if not df.empty and "model_name" in df.columns and "epochs" in df.columns:
        # Keep the most-trained (highest-epoch) run per model_name in case of reruns.
        df = df.sort_values("epochs").drop_duplicates(subset="model_name", keep="last").reset_index(drop=True)
    return df


CLASSIFICATION_PHASE_DIRS = [
    RESULTS_ROOT / "phase_1_baseline_training",
    RESULTS_ROOT / "phase_2_kernel_restriction_training",
    RESULTS_ROOT / "phase_3_compensation_and_hybrids_training",
    RESULTS_ROOT / "phase_4_compression_and_final_architecture_training",
]
PHASE9_DIR = PROJECT_ROOT / "outputs" / "pcad" / "phase_9_bypass_ablation"
PHASE8_CSV = RESULTS_ROOT / "phase_8_efficient_vit_hybrid_attention_analysis" / "phase8_comparison.csv"

frames = [load_summary_jsons(d) for d in CLASSIFICATION_PHASE_DIRS]
frames.append(load_summary_jsons(PHASE9_DIR))

if PHASE8_CSV.exists():
    df_phase8 = pd.read_csv(PHASE8_CSV)
    if "phase" not in df_phase8.columns:
        df_phase8["phase"] = "Phase 8 — Efficient ViT / Hybrid-Attention"
    frames.append(df_phase8)
else:
    print(f"[skip] {PHASE8_CSV} not found")

SHARED_COLS = [
    "phase", "model_name", "fp32_top1", "fp32_top5", "int8_top1", "int8_top5",
    "quantization_drop_top1", "fp32_size_mb", "int8_size_mb", "params_m", "macs",
    "fp32_latency_ms_per_image", "fp32_throughput_img_per_s",
    "param_efficiency_top1_per_m", "total_training_time_s",
    "avg_epoch_time_s", "peak_gpu_mem_mb",  # computed by make_run_summary but unused until now
]
frames = [f for f in frames if not f.empty]
df_all_classification = pd.concat(
    [f[[c for c in SHARED_COLS if c in f.columns]] for f in frames],
    ignore_index=True,
) if frames else pd.DataFrame(columns=SHARED_COLS)
df_all_classification = df_all_classification.drop_duplicates(subset="model_name", keep="last").reset_index(drop=True)

PHASE_ORDER = sorted(df_all_classification["phase"].dropna().unique().tolist()) if not df_all_classification.empty else []

print(f"Loaded {len(df_all_classification)} classification models across {len(PHASE_ORDER)} phases.")
display(df_all_classification[["phase", "model_name", "fp32_top1", "int8_top1", "quantization_drop_top1",
                                "fp32_size_mb", "params_m"]].sort_values("fp32_top1", ascending=False).reset_index(drop=True))


## Load Detection & Segmentation Results (Phase 7)

Loaded straight from the CSVs `notebooks/phase_7_detection_segmentation_analysis/phase7_results_analysis.ipynb`
already produced. Only `status == "complete"` and `post_fix == True` rows are kept (`post_fix`
marks runs built after the anchor-recall bug fix — see that notebook for details).

In [ ]:
def load_phase7_csv(name: str) -> pd.DataFrame:
    path = RESULTS_ROOT / "phase_7_detection_segmentation_analysis" / name
    if not path.exists():
        print(f"[skip] {path} not found")
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "status" in df.columns:
        df = df[df["status"] == "complete"]
    if "post_fix" in df.columns:
        df = df[df["post_fix"] == True]  # noqa: E712
    return df.reset_index(drop=True)


df_detection = load_phase7_csv("phase7_comparison.csv")
df_segmentation = load_phase7_csv("phase7_seg_comparison.csv")
print(f"Detection runs (complete, post-fix): {len(df_detection)}")
print(f"Segmentation runs (complete, post-fix): {len(df_segmentation)}")


## Load Hardware Profiling Highlights (Phase 6)

Phase 6 measures latency/power on already-trained Phase 1-4 models rather than training new ones, so it doesn't add rows to the classification ranking above — just a compute-efficiency-by-kernel-group check (Winograd evidence) and a latency/accuracy view. See `notebooks/phase_6_hardware_profiling_analysis/hardware_profiling_phase6.ipynb` for the full H1-H4 hypothesis analysis.

In [ ]:
PHASE6_DIR = RESULTS_ROOT / "phase_6_hardware_profiling_analysis"

def load_phase6_csv(name: str) -> pd.DataFrame:
    path = PHASE6_DIR / name
    if not path.exists():
        print(f"[skip] {path} not found")
        return pd.DataFrame()
    return pd.read_csv(path)


df_winograd_evidence = load_phase6_csv("h1_winograd_evidence.csv")
df_latency_pareto = load_phase6_csv("h3_latency_pareto.csv")
print(f"Winograd evidence rows: {len(df_winograd_evidence)}")
print(f"Latency/accuracy rows: {len(df_latency_pareto)}")


## 1. All-Model FP32 Top-1 Ranking

In [ ]:
if not df_all_classification.empty:
    rank_df = df_all_classification[df_all_classification["fp32_top1"].notna()] \
        .sort_values("fp32_top1").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(rank_df))))
    ax.barh(rank_df["model_name"], rank_df["fp32_top1"],
            color=phase_colors_for(rank_df["phase"], PHASE_ORDER))
    ax.set_xlabel("FP32 Top-1 Accuracy (%)")
    ax.set_title(f"All Trained Models — FP32 Top-1 Ranking ({len(rank_df)} models, Phases 1-4/8/9)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=PHASE_CMAP(i % 10)) for i in range(len(PHASE_ORDER))]
    ax.legend(handles, PHASE_ORDER, loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "all_models_fp32_ranking.png")
    plt.show()
else:
    print("[skip] No classification data loaded")


## 2. Accuracy vs. Model Size — Pareto Frontier (All Classification Models)

In [ ]:
def pareto_front_mask(xs, ys):
    """Boolean mask: True if point is Pareto-optimal (minimize x, maximize y)."""
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    dominated = np.zeros(len(xs), dtype=bool)
    for i in range(len(xs)):
        for j in range(len(xs)):
            if i != j and xs[j] <= xs[i] and ys[j] >= ys[i] and (xs[j] < xs[i] or ys[j] > ys[i]):
                dominated[i] = True
                break
    return ~dominated


df_size = df_all_classification[df_all_classification["fp32_top1"].notna()
                                 & df_all_classification["fp32_size_mb"].notna()].copy()
if len(df_size) >= 2:
    mask = pareto_front_mask(df_size["fp32_size_mb"].values, df_size["fp32_top1"].values)
    pf = df_size[mask].sort_values("fp32_size_mb")

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.scatter(df_size["fp32_size_mb"], df_size["fp32_top1"],
               c=phase_colors_for(df_size["phase"], PHASE_ORDER), s=120,
               edgecolors="white", lw=0.5, alpha=0.9, zorder=3)
    ax.step(pf["fp32_size_mb"], pf["fp32_top1"], where="post",
            color="black", lw=1.2, ls="--", alpha=0.6, zorder=2)
    for _, row in pf.iterrows():
        ax.annotate(row["model_name"], (row["fp32_size_mb"], row["fp32_top1"]),
                    xytext=(6, 6), textcoords="offset points", fontsize=8, fontweight="bold")
    ax.set_xscale("log")
    ax.set_xlabel("FP32 Model Size (MB, log scale)")
    ax.set_ylabel("FP32 Top-1 Accuracy (%)")
    ax.set_title(f"Accuracy vs. Size — All Classification Models (Pareto front: {len(pf)}/{len(df_size)})")
    handles = [plt.Rectangle((0, 0), 1, 1, color=PHASE_CMAP(i % 10)) for i in range(len(PHASE_ORDER))]
    ax.legend(handles, PHASE_ORDER, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, title="Phase")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "all_models_accuracy_vs_size_pareto.png", bbox_inches="tight")
    plt.show()
else:
    print("[skip] Not enough models with size + accuracy")


## 3. Quantization Drop (FP32 → INT8) — All Classification Models

In [ ]:
df_drop = df_all_classification[df_all_classification["quantization_drop_top1"].notna()] \
    .sort_values("quantization_drop_top1").reset_index(drop=True)
if not df_drop.empty:
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(df_drop))))
    ax.barh(df_drop["model_name"], df_drop["quantization_drop_top1"],
            color=phase_colors_for(df_drop["phase"], PHASE_ORDER))
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("FP32 → INT8 Top-1 Drop (pp)  —  negative is worse")
    ax.set_title(f"Quantization Robustness — All Classification Models ({len(df_drop)} models)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=PHASE_CMAP(i % 10)) for i in range(len(PHASE_ORDER))]
    ax.legend(handles, PHASE_ORDER, loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "all_models_quantization_drop.png")
    plt.show()
else:
    print("[skip] No quantization-drop data")


## 3b. Training Efficiency — Avg Epoch Time (All Classification Models)

`avg_epoch_time_s` is computed by `make_run_summary` for every run but wasn't previously surfaced
here. Per-epoch *curves* (loss/accuracy over training, LR schedule) are a different notebook —
see `training_dynamics.ipynb` — since the source data (raw logs/TensorBoard events) and only
partial phase coverage make it a poor fit for this notebook's one-row-per-model tables.

In [ ]:
df_epoch_time = df_all_classification[df_all_classification["avg_epoch_time_s"].notna()] \
    .sort_values("avg_epoch_time_s").reset_index(drop=True)
if not df_epoch_time.empty:
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(df_epoch_time))))
    ax.barh(df_epoch_time["model_name"], df_epoch_time["avg_epoch_time_s"],
            color=phase_colors_for(df_epoch_time["phase"], PHASE_ORDER))
    ax.set_xlabel("Avg Epoch Time (s)")
    ax.set_title(f"Training Efficiency — Avg Epoch Time ({len(df_epoch_time)} models)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=PHASE_CMAP(i % 10)) for i in range(len(PHASE_ORDER))]
    ax.legend(handles, PHASE_ORDER, loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "all_models_avg_epoch_time.png")
    plt.show()
else:
    print("[skip] No avg_epoch_time_s data")


## 4. Detection — mAP vs. Model Size (Phase 7)

In [ ]:
STAGE_COLORS = {"fp32": "#4a3aa7", "qat": "#eb6834", "int8": "#1baf7a"}

if not df_detection.empty and "true_size_mb" in df_detection.columns:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for stage, color in STAGE_COLORS.items():
        sv = df_detection[(df_detection["stage"] == stage) & df_detection["best_val_mAP"].notna()
                           & df_detection["true_size_mb"].notna()]
        if sv.empty:
            continue
        ax.scatter(sv["true_size_mb"], sv["best_val_mAP"], color=color, s=120,
                   label=stage.upper(), edgecolors="white", lw=0.5, zorder=3)
        for _, row in sv.iterrows():
            ax.annotate(row["model"], (row["true_size_mb"], row["best_val_mAP"]),
                        xytext=(6, 4), textcoords="offset points", fontsize=7.5)
    ax.set_xlabel("True Deployable Model Size (MB)")
    ax.set_ylabel("Detection mAP")
    ax.set_title("Phase 7 Detection — mAP vs. Model Size")
    ax.legend(fontsize=9, title="Stage")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "detection_map_vs_size.png")
    plt.show()
else:
    print("[skip] No detection size/accuracy data")


## 5. Segmentation — mIoU vs. Model Size (Phase 7)

In [ ]:
if not df_segmentation.empty and "true_size_mb" in df_segmentation.columns:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for stage, color in STAGE_COLORS.items():
        sv = df_segmentation[(df_segmentation["stage"] == stage) & df_segmentation["best_val_mIoU"].notna()
                              & df_segmentation["true_size_mb"].notna()]
        if sv.empty:
            continue
        ax.scatter(sv["true_size_mb"], sv["best_val_mIoU"], color=color, s=120,
                   label=stage.upper(), edgecolors="white", lw=0.5, zorder=3)
        for _, row in sv.iterrows():
            ax.annotate(row["model"], (row["true_size_mb"], row["best_val_mIoU"]),
                        xytext=(6, 4), textcoords="offset points", fontsize=7.5)
    ax.set_xlabel("True Deployable Model Size (MB)")
    ax.set_ylabel("Segmentation mIoU")
    ax.set_title("Phase 7 Segmentation — mIoU vs. Model Size")
    ax.legend(fontsize=9, title="Stage")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "segmentation_miou_vs_size.png")
    plt.show()
else:
    print("[skip] No segmentation size/accuracy data")


## 6. Hardware Profiling Highlights (Phase 6)

In [ ]:
if not df_winograd_evidence.empty or not df_latency_pareto.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    if not df_winograd_evidence.empty and "group" in df_winograd_evidence.columns:
        g = df_winograd_evidence.groupby("group")["compute_efficiency_gflops_s"].mean().sort_values()
        axes[0].barh(g.index, g.values, color=PHASE_CMAP(5))
        axes[0].set_xlabel("Mean Compute Efficiency (GFLOPs/s)")
        axes[0].set_title("Winograd Evidence — Efficiency by Kernel Group")
    else:
        axes[0].axis("off")
        axes[0].set_title("[skip] No Winograd evidence data")

    if not df_latency_pareto.empty and all(c in df_latency_pareto.columns for c in ("latency_ms", "accuracy")):
        for precision in df_latency_pareto["precision"].dropna().unique():
            sv = df_latency_pareto[df_latency_pareto["precision"] == precision]
            axes[1].scatter(sv["latency_ms"], sv["accuracy"], s=100, label=str(precision), alpha=0.85)
        axes[1].set_xlabel("Latency (ms)")
        axes[1].set_ylabel("Accuracy (%)")
        axes[1].set_title("Latency vs. Accuracy, by Precision")
        axes[1].legend(fontsize=8, title="Precision")
    else:
        axes[1].axis("off")
        axes[1].set_title("[skip] No latency/accuracy data")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "phase6_hardware_highlights.png")
    plt.show()
else:
    print("[skip] No Phase 6 data loaded")


## Master Tables

In [ ]:
print("=== Classification — all models, ranked by FP32 top-1 ===")
display(df_all_classification.sort_values("fp32_top1", ascending=False).reset_index(drop=True))

if not df_detection.empty:
    print("\n=== Detection — ranked by mAP ===")
    display(df_detection.sort_values("best_val_mAP", ascending=False).reset_index(drop=True))

if not df_segmentation.empty:
    print("\n=== Segmentation — ranked by mIoU ===")
    display(df_segmentation.sort_values("best_val_mIoU", ascending=False).reset_index(drop=True))


## Conclusions

Generated from the tables loaded above, so it stays correct as new Phase 8/9 results land — not a static write-up.

In [ ]:
lines = []

if not df_all_classification.empty and df_all_classification["fp32_top1"].notna().any():
    m, p, v = df_all_classification.loc[df_all_classification["fp32_top1"].idxmax(), ["model_name", "phase", "fp32_top1"]]
    lines.append(f"- Best FP32 accuracy overall: **{m}** ({p}) at {v:.2f}% top-1.")

if not df_all_classification.empty and df_all_classification["fp32_top1"].notna().any() \
        and df_all_classification["fp32_size_mb"].notna().any():
    eff = df_all_classification[df_all_classification["fp32_top1"].notna()
                                 & df_all_classification["fp32_size_mb"].notna()].copy()
    eff["acc_per_mb"] = eff["fp32_top1"] / eff["fp32_size_mb"]
    best_eff = eff.loc[eff["acc_per_mb"].idxmax()]
    m, p, a, acc, sz = best_eff["model_name"], best_eff["phase"], best_eff["acc_per_mb"], best_eff["fp32_top1"], best_eff["fp32_size_mb"]
    lines.append(f"- Best accuracy-per-MB efficiency: **{m}** ({p}) at {a:.2f}%/MB ({acc:.2f}% in {sz:.2f} MB).")

if not df_all_classification.empty and df_all_classification["quantization_drop_top1"].notna().any():
    most_stable = df_all_classification.loc[df_all_classification["quantization_drop_top1"].idxmax()]
    m, p, d = most_stable["model_name"], most_stable["phase"], most_stable["quantization_drop_top1"]
    lines.append(f"- Most quantization-robust: **{m}** ({p}), FP32\u2192INT8 change of {d:+.2f}pp.")

if not df_detection.empty and df_detection["best_val_mAP"].notna().any():
    best_det = df_detection.loc[df_detection["best_val_mAP"].idxmax()]
    m, s, v = best_det["model"], best_det["stage"], best_det["best_val_mAP"]
    lines.append(f"- Best detection result: **{m}** ({s}) at {v:.3f} mAP.")

if not df_segmentation.empty and df_segmentation["best_val_mIoU"].notna().any():
    best_seg = df_segmentation.loc[df_segmentation["best_val_mIoU"].idxmax()]
    m, s, v = best_seg["model"], best_seg["stage"], best_seg["best_val_mIoU"]
    lines.append(f"- Best segmentation result: **{m}** ({s}) at {v:.3f} mIoU.")

print("\n".join(lines) if lines else "[skip] Not enough data loaded to summarize.")


### Limitations

- Detection (mAP) and segmentation (mIoU) are not on the same scale as classification top-1 and are never merged into one ranking with it.
- Phase 6 measures hardware behavior of already-listed Phase 1-4 models, so it contributes no new rows to the classification ranking.
- Phase 8's `vit_tiny`/`deit_tiny` (trained via `notebooks/phase_8_efficient_vit/vit_qat_phase8.ipynb`, not the CLI) only appear here once that notebook has produced their summary artifacts — nothing is hardcoded, so re-running this notebook later picks them up automatically.

## Persist Results

In [ ]:
df_all_classification.to_csv(RESULTS_DIR / "all_classification_models.csv", index=False)
if not df_detection.empty:
    df_detection.to_csv(RESULTS_DIR / "all_detection_models.csv", index=False)
if not df_segmentation.empty:
    df_segmentation.to_csv(RESULTS_DIR / "all_segmentation_models.csv", index=False)

executive_summary = {
    "title": "Phase 10 — Final Cross-Project Summary",
    "phases_covered": PHASE_ORDER,
    "n_classification_models": int(len(df_all_classification)),
    "n_detection_runs": int(len(df_detection)),
    "n_segmentation_runs": int(len(df_segmentation)),
    "best_fp32_overall": (
        df_all_classification.loc[df_all_classification["fp32_top1"].idxmax()].to_dict()
        if not df_all_classification.empty and df_all_classification["fp32_top1"].notna().any() else None
    ),
    "most_quantization_stable": (
        df_all_classification.loc[df_all_classification["quantization_drop_top1"].idxmax()].to_dict()
        if not df_all_classification.empty and df_all_classification["quantization_drop_top1"].notna().any() else None
    ),
    "best_detection_mAP": (
        df_detection.loc[df_detection["best_val_mAP"].idxmax()].to_dict()
        if not df_detection.empty and df_detection["best_val_mAP"].notna().any() else None
    ),
    "best_segmentation_mIoU": (
        df_segmentation.loc[df_segmentation["best_val_mIoU"].idxmax()].to_dict()
        if not df_segmentation.empty and df_segmentation["best_val_mIoU"].notna().any() else None
    ),
}

out_path = RESULTS_DIR / "executive_summary.json"
with open(out_path, "w") as f:
    json.dump(executive_summary, f, indent=2, default=str)
print(f"Saved: {out_path}")
